# 01 · Generación del Dataset
## Proyecto StyleNow Analytics
Este notebook genera el dataset sucio de ventas de la tienda e-commerce StyleNow.

In [2]:
# Importamos las librerías que vamos a usar
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [7]:
# Fijamos una semilla para reproducibilidad
np.random.seed(42) 
random.seed(42)

# Numero de pedidos que tendrá nuestro dataset
n = 100000

# Lista de valores posibles
categorias =  ['Camisetas', 'Pantalones', 'Vestidos', 'Zapatos', 'Accesorios']

productos = {
    'Camisetas':  ['Camiseta Básica', 'Polo Slim', 'Camiseta Estampada', 'Camiseta Oversize', 'Top Tirantes'],
    'Pantalones': ['Vaquero Slim', 'Chino Beige', 'Pantalón Casual', 'Jogger Negro', 'Vaquero Wide Leg'],
    'Vestidos':   ['Vestido Floral', 'Vestido Negro', 'Vestido Verano', 'Vestido Midi', 'Vestido Lencero'],
    'Zapatos':    ['Zapatilla Blanca', 'Bota Marrón', 'Sandalia Plana', 'Tacón Nude', 'Deportiva Runner'],
    'Accesorios': ['Bolso Mini', 'Cinturón Cuero', 'Gafas Sol', 'Sombrero Paja', 'Bufanda Lana']
}

paises = ['España', 'Francia', 'Italia', 'Alemania', 'Portugal']

# Generamos fechas aleatorias entre 2023 y 2024
fecha_inicio = datetime(2023,1,1)
fechas = [fecha_inicio + timedelta(days=random.randint(0,1095)) for _ in range(n)]

#Generamos cada columna
cat_lista = [random.choice(categorias) for _ in range(n)]
prod_lista = [random.choice(productos[cat]) for cat in cat_lista]

precio_base = {'Camisetas': 25, 'Pantalones': 55, 'Vestidos': 75, 'Zapatos': 90, 'Accesorios': 35}
precios = [round(precio_base[c] + np.random.normal(0,8),2) for c in cat_lista]

descuentos  = [round(random.choice([0, 0, 0, 0, 5, 10, 15, 20, 25]), 2) for _ in range(n)]
clientes    = [f'CLI{random.randint(1000, 1500):04d}' for _ in range(n)]
pedidos     = [f'PED{i+10000:05d}' for i in range(n)]
cantidades  = [random.randint(1, 4) for _ in range(n)]

In [8]:
# Contruimos el dataframe
df = pd.DataFrame({
    'id_pedido': pedidos,
    'id_cliente': clientes,
    'fecha': fechas,
    'categoria': cat_lista,
    'producto': prod_lista,
    'precio': precios,
    'descuento': descuentos,
    'cantidad': cantidades,
    'pais' :  [random.choice(paises) for _ in range(n)]
})

print(f"Dataset creado con {df.shape[0]} filas y {df.shape[1]} columnas")
print(df.head())

Dataset creado con 100000 filas y 9 columnas
  id_pedido id_cliente      fecha   categoria           producto  precio  \
0  PED10000    CLI1162 2023-08-17  Accesorios          Gafas Sol   38.97   
1  PED10001    CLI1194 2023-02-21   Camisetas          Polo Slim   23.89   
2  PED10002    CLI1097 2024-07-17   Camisetas  Camiseta Oversize   30.18   
3  PED10003    CLI1327 2024-05-16     Zapatos        Bota Marrón  102.18   
4  PED10004    CLI1440 2024-04-02     Zapatos         Tacón Nude   88.13   

   descuento  cantidad      pais  
0         20         2    Italia  
1          5         4    Italia  
2         25         1   Francia  
3          0         3  Portugal  
4          0         4   Francia  


## Introducimos suciedad al dataset
Simulamos los errores típicos que ocurren en datos reales de empresa.

In [10]:
#  Hacemos una copia del dataframe limpio antes de ensuaciarlo
df_sucio = df.copy()

In [21]:
# SUCIEDAD 1: Nulos aleatorios en columnas clave
# En la vida real ocurre por formularios incompletos o errores de integración entre sistemas

for col in ['precio', 'descuento', 'pais', 'categoria']:
    idx = df_sucio.sample(frac=0.01).index
    df_sucio.loc[idx, col] = np.nan

print("Nulos introducidos")
print(df_sucio.isnull().sum())

Nulos introducidos
id_pedido        0
id_cliente       0
fecha            0
categoria     1000
producto         0
precio        1000
descuento     1000
cantidad         0
pais          1000
dtype: int64


In [22]:
# SUCIEDAD 2: Duplicados
# En la vida real ocurren por doble click en el checkout,
# fallos de red o errores al fusionar datasets de distintos sistemas

duplicados = df_sucio.sample(200)
df_sucio = pd.concat([df_sucio, duplicados], ignore_index=True)

print(f"Filas después de añadir duplicados: {df_sucio.shape[0]}")

Filas después de añadir duplicados: 100200


In [23]:
# SUCIEDAD 3: Errores de texto en categoría
# En la vida real ocurre cuando distintos sistemas exportan
# el mismo valor de formas diferentes

# Algunas categorías en mayúsculas
idx_mayus = df_sucio.sample(frac=0.03).index
df_sucio.loc[idx_mayus, 'categoria'] = df_sucio.loc[idx_mayus, 'categoria'].str.upper()

# Algunas categorías con espacio delante
idx_espacio = df_sucio.sample(frac=0.02).index
df_sucio.loc[idx_espacio, 'categoria'] = ' ' + df_sucio.loc[idx_espacio, 'categoria']

print("Valores únicos en categoría después de los errores:")
print(df_sucio['categoria'].unique())

Valores únicos en categoría después de los errores:
['Accesorios' 'Camisetas' 'Zapatos' 'Pantalones' 'Vestidos' ' Vestidos'
 ' Accesorios' 'ACCESORIOS' 'PANTALONES' ' Pantalones' ' CAMISETAS'
 ' Zapatos' nan ' Camisetas' 'CAMISETAS' 'VESTIDOS' 'ZAPATOS'
 ' PANTALONES' ' VESTIDOS' ' ACCESORIOS' ' ZAPATOS']


In [24]:
# SUCIEDAD 4: Precios negativos
# Ocurre por bugs en sistemas de devoluciones o errores manuales

idx_precio = df_sucio.sample(10).index
df_sucio.loc[idx_precio, 'precio'] = -abs(df_sucio.loc[idx_precio, 'precio'])

print("Precios negativos introducidos:")
print(df_sucio[df_sucio['precio'] < 0][['id_pedido', 'producto', 'precio']])

Precios negativos introducidos:
      id_pedido            producto  precio
1531   PED11531      Vestido Floral  -60.85
3810   PED13810      Vestido Floral  -67.05
8024   PED18024           Polo Slim   -0.11
8180   PED18180     Camiseta Básica   -0.93
15857  PED25857        Top Tirantes  -41.58
21421  PED31421     Camiseta Básica   -0.70
31304  PED41304   Camiseta Oversize   -0.63
33667  PED43667           Polo Slim   -2.49
34500  PED44500        Vaquero Slim  -43.67
42904  PED52904    Deportiva Runner -108.20
45312  PED55312  Camiseta Estampada   -0.90
53591  PED63591  Camiseta Estampada   -1.12
53868  PED63868           Polo Slim   -2.36
54702  PED64702   Camiseta Oversize   -1.23
67989  PED77989        Top Tirantes  -33.15
74459  PED84459          Tacón Nude  -98.56
74854  PED84854           Polo Slim   -0.57
75139  PED85139        Bufanda Lana  -29.80
76242  PED86242        Top Tirantes   -2.80
76830  PED86830      Cinturón Cuero  -20.48
77265  PED87265  Camiseta Estampada   -0.46


In [28]:
# SUCIEDAD 5: Fechas en formato string incorrecto
# Ocurre al exportar desde sistemas legacy o Excel mal configurado
#  Primero convertimos la columna fecha a string
df_sucio['fecha'] = df_sucio['fecha'].astype(str)

# Ahora sí introducimos el valor incorrecto
idx_fecha = df_sucio.sample(frac=0.02).index
df_sucio.loc[idx_fecha, 'fecha'] = 'sin fecha'

print(df_sucio[df_sucio['fecha'] == 'sin fecha'].shape[0], "filas con fecha incorrecta")

7791 filas con fecha incorrecta


In [29]:
# SUCIEDAD 6: Descuentos mayores al 100%
# Ocurre por errores de entrada manual de datos

idx_desc = df_sucio.sample(15).index
df_sucio.loc[idx_desc, 'descuento'] = 150

print("Descuentos imposibles introducidos:")
print(df_sucio[df_sucio['descuento'] > 100][['id_pedido', 'producto', 'descuento']])

Descuentos imposibles introducidos:
      id_pedido           producto  descuento
1177   PED11177   Deportiva Runner      150.0
3776   PED13776    Vestido Lencero      150.0
5176   PED15176      Sombrero Paja      150.0
11516  PED21516  Camiseta Oversize      150.0
12861  PED22861     Sandalia Plana      150.0
20172  PED30172      Sombrero Paja      150.0
24011  PED34011   Vaquero Wide Leg      150.0
36192  PED46192     Vestido Floral      150.0
42726  PED52726         Bolso Mini      150.0
56395  PED66395     Vestido Floral      150.0
57272  PED67272        Chino Beige      150.0
63751  PED73751   Deportiva Runner      150.0
71144  PED81144   Zapatilla Blanca      150.0
72297  PED82297    Vestido Lencero      150.0
80351  PED90351    Vestido Lencero      150.0


In [33]:
# Guardamos el dataset sucio en data/raw/
# Recuerda: data/raw es donde vive el dato original sin tocar

df_sucio.to_csv('../data/raw/stylenow_raw.csv', index=False)

print(f"Dataset guardado correctamente")
print(f"Filas totales: {df_sucio.shape[0]}")
print(f"Columnas: {df_sucio.shape[1]}")
print(f"\nResumen de suciedad introducida:")
print(f"- Nulos por columna: ~1.000")
print(f"- Duplicados: 200")
print(f"- Errores de texto en categoría: ~5.000")
print(f"- Precios negativos: ~21")
print(f"- Fechas incorrectas: 7.791")
print(f"- Descuentos imposibles: 15")

Dataset guardado correctamente
Filas totales: 100200
Columnas: 9

Resumen de suciedad introducida:
- Nulos por columna: ~1.000
- Duplicados: 200
- Errores de texto en categoría: ~5.000
- Precios negativos: ~21
- Fechas incorrectas: 7.791
- Descuentos imposibles: 15
